# 🏢 Enterprise AP-Env: Training Notebook

**Meta AI Hackathon Grand Finale**

This notebook trains a rule-based agent on the Enterprise Accounts Payable environment across all 5 task difficulties and produces reward curves showing agent improvement.

## What this environment does
An AI agent is dropped into an enterprise AP department and must:
- Read emails from an inbox to find invoices
- Query an ERP system (with schema drift)
- Extract structured fields from unstructured invoice text
- Detect anomalies: price mismatches, tax fraud, duplicate invoices, phishing
- Negotiate with vendors via email
- Make a final approve/reject decision

## Tasks
| Task | Difficulty | Description |
|------|-----------|-------------|
| `easy` | Beginner | Read email → Query ERP → Extract fields → Approve |
| `medium` | Medium | Detect price mismatch → Reject |
| `hard` | Hard | Handle ERP schema drift + detect duplicate invoice |
| `expert_negotiation` | Expert | Email vendor, get corrected invoice, approve |
| `expert_fraud` | Expert | Detect lookalike domain phishing attack |

## Step 1: Install Dependencies

In [ ]:
!pip install fastapi uvicorn pydantic openai python-dotenv requests gradio matplotlib --quiet
print('✅ Dependencies installed')

## Step 2: Clone the Repository

In [ ]:
import os

if not os.path.exists('Enterprise-AP-Environment'):
    !git clone https://github.com/dharmendra26-wiz/Enterprise-AP-Environment
    print('✅ Repo cloned')
else:
    !cd Enterprise-AP-Environment && git pull
    print('✅ Repo updated')

os.chdir('Enterprise-AP-Environment')
print(f'📁 Working directory: {os.getcwd()}')

## Step 3: Verify the Environment Works

In [ ]:
import sys
sys.path.insert(0, '.')

from app.environment import EnterpriseAPEnvironment
from app.models import Action

# Quick smoke test
for task in ['easy', 'medium', 'hard', 'expert_negotiation', 'expert_fraud']:
    env = EnterpriseAPEnvironment(task)
    obs = env.reset()
    print(f'✅ {task:25s} | inbox={len(obs.inbox_status)} emails | step={obs.current_step}')

print('\n🎉 All 5 tasks initialized successfully!')

## Step 4: Run Training (60 Episodes per Task)

In [ ]:
!python train.py --episodes 60

## Step 5: Display Reward Curves

In [ ]:
from IPython.display import Image, display
display(Image('reward_curves.png', width=900))

## Step 6: Show Final Scores

In [ ]:
import json

with open('training_results.json') as f:
    results = json.load(f)

print('=' * 50)
print('  FINAL TRAINING RESULTS')
print('=' * 50)
targets = {'easy': 0.85, 'medium': 0.75, 'hard': 0.65, 'expert_negotiation': 0.70, 'expert_fraud': 0.70}
all_pass = True
for task, score in results['final_scores'].items():
    target = targets[task]
    status = 'PASS' if score >= target else 'FAIL'
    if score < target:
        all_pass = False
    print(f'  [{status}] {task:<25} score={score:.3f}  target={target}')

print('=' * 50)
if all_pass:
    print('  🎉 ALL TASKS PASSED!')
else:
    print('  ⚠️  Some tasks below target - try more episodes')

## Step 7: Run a Single Episode Manually
Watch the agent take actions step by step on the `expert_fraud` task:

In [ ]:
import re, sys
sys.path.insert(0, '.')
from app.environment import EnterpriseAPEnvironment
from app.models import Action

def parse_vendor(body):
    m = re.search(r'Vendor:\s*(.+)', body, re.IGNORECASE)
    return m.group(1).strip() if m else 'Unknown'

task_name = 'expert_fraud'
env = EnterpriseAPEnvironment(task_name)
obs = env.reset()
print(f'\n--- Running {task_name} episode ---')
print(f'Inbox: {[e["sender"] for e in obs.inbox_status]}')
print(f'\nNOTICE: Check if the sender domain looks suspicious!')

# Step 1: Read email
r = env.step(Action(action_type='read_email', email_id=obs.inbox_status[0]['id']))
print(f'\nStep 1 - READ EMAIL: reward={r.reward}')

# Step 2: Query ERP
vendor = parse_vendor(r.observation.email_content or '')
r = env.step(Action(action_type='query_erp', api_endpoint='/api/v1/po', api_payload={'vendor_name': vendor}))
print(f'Step 2 - QUERY ERP ({vendor}): reward={r.reward}')

# Step 3: Flag as fraud
r = env.step(Action(action_type='flag', field_name='fraud'))
print(f'Step 3 - FLAG FRAUD: reward={r.reward}')

# Step 4: Reject
r = env.step(Action(action_type='reject'))
print(f'Step 4 - REJECT: final_score={r.reward}')
print(f'\nResult: {r.observation.message}')

## Step 8: Real LLM Training with TRL GRPO & Unsloth

This section demonstrates how to connect our environment to a real gradient-based training loop using Hugging Face TRL and Unsloth. We fine-tune Qwen-0.5B using GRPO (Group Relative Policy Optimization) on the `easy` task as a minimal proof of concept.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes --quiet
print('✅ Unsloth & TRL installed')

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from app.environment import EnterpriseAPEnvironment
from app.models import Action
import json

# 1. Load Model
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
)

# 2. Define Env Reward Function for TRL
def env_reward_func(completions, prompts):
    rewards = []
    env = EnterpriseAPEnvironment("easy")
    
    for completion in completions:
        env.reset()
        text = completion[0]['content']
        try:
            import re
            m = re.search(r"\{.*\}", text, re.DOTALL)
            if m:
                action_dict = json.loads(m.group())
                result = env.step(Action(**action_dict))
                rewards.append(result.reward)
            else:
                rewards.append(-0.1)
        except Exception:
            rewards.append(-0.1)
            
    return rewards

# 3. Dummy Dataset (Prompts to trigger policy generation)
from datasets import Dataset
dummy_dataset = Dataset.from_dict({
    "prompt": [[
        {"role": "system", "content": "You are an AP Agent. Respond with a JSON action."},
        {"role": "user", "content": "Task: easy. Your inbox:\n [email_001] From: vendor | Subject: Invoice attached\nBegin processing."}
    ]] * 16
})

# 4. Train
training_args = GRPOConfig(
    output_dir="outputs",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=256,
    max_completion_length=256,
    num_generations=4,
    max_steps=10,
    logging_steps=1,
    optim="adamw_8bit",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[env_reward_func],
    args=training_args,
    train_dataset=dummy_dataset,
)

print("Starting GRPO training step...")
trainer.train()

## Summary

This environment trains agents to handle real-world enterprise workflows:
- **Multi-app navigation**: Email + ERP + Vendor communication
- **Schema drift adaptation**: ERP API changes its contract mid-workflow  
- **Multi-agent negotiation**: Agent emails vendor, vendor replies with corrected invoice
- **Security/fraud detection**: Lookalike phishing domain detection

Built for the **Meta AI Hackathon Grand Finale 2026** 🏆